# Linked Views: Building an Interactive Bird-Observation Dashboard with Bokeh

*A worked example in coordinated multiple views — where selecting in one chart reshapes the others, and the interaction itself carries the insight.*

---

Most dashboards are a grid of charts that happen to share a page. You read each one in isolation, do the cross-referencing in your head, and hope you remember what the third chart said while you look at the first. This tutorial is about the alternative: **coordinated multiple views**, where the charts are wired together so that a selection in one propagates to all of them. The analyst brushes a region on a map and every other panel instantly re-expresses itself for just those points.

That shift — from *reading* charts to *interrogating* them — is the whole point, and it changes how you should design each individual view.

We'll build a four-chart dashboard over recent Michigan bird observations from [eBird](https://ebird.org), and along the way cover:

1. **Each visualization type** — what question it answers, and how to read it
2. **How the four complement each other** — and the interactivity that ties them together
3. **The framework** — Bokeh: who made it, licensing, install, Jupyter integration
4. **The framework's approach and limits** — its model, and why it over the alternatives here
5. **The dataset** — pulling from the eBird API and cleaning it

The finished artifact is a single self-contained HTML file with no server behind it — publishable anywhere, interactive for anyone who opens it.

## 3. The framework: Bokeh

*(Taken out of order — the tooling choice frames everything that follows.)*

[**Bokeh**](https://bokeh.org) is an open-source Python visualization library, created by Peter Wang, Hugo Shi, and colleagues at Continuum Analytics (now Anaconda) starting in 2012, with initial funding from a DARPA grant. It is maintained today by [NumFOCUS](https://numfocus.org) as a fiscally sponsored project and is released under the permissive **BSD 3-Clause license**, so it is free for commercial use.

Its defining idea: **you write Python, Bokeh emits a JSON scene graph, and BokehJS renders and drives it in the browser.** The interactivity — pan, zoom, hover, and crucially *linked selection* — runs client-side. That means a dashboard can be exported to a static HTML file and stay fully interactive with no Python process alive behind it. For a tutorial others should be able to open and poke at, that property is decisive.

Install it from PyPI or conda:

```bash
pip install bokeh        # or:  conda install bokeh
```

It ships with no heavy system dependencies — just NumPy, Pandas, Pillow, Jinja2, and friends. In a notebook, one call wires the rich display hooks so figures render inline as live BokehJS, not static images:

In [1]:
import numpy as np
import pandas as pd
import requests
from datetime import date, timedelta

import bokeh
from bokeh.io import output_notebook, show
from bokeh.plotting import figure

output_notebook()   # route Bokeh output into the notebook's cells
print("Bokeh", bokeh.__version__)

Loading BokehJS ...

Bokeh 3.9.1


## 4. How Bokeh works, and why it fits here

**It sits between declarative and procedural.** At the low level you assemble a figure imperatively — add a glyph, attach a tool, wire a callback. But the objects you build (`ColumnDataSource`, `figure`, `CustomJS`) form a declarative *model*: you describe the scene and its data bindings, and BokehJS decides how to render and update it. This matters for linked views because "when this selection changes, recompute that source" is expressed as a declared relationship, not a redraw loop you manage yourself.

**The `ColumnDataSource` is the keystone.** It is a columnar data structure shared by reference across glyphs. Two charts backed by the same source share a selection automatically — brush points in one and the corresponding rows highlight in the other, for free. When two charts are backed by *different* sources (a per-observation scatter and a per-species bar chart, say), you bridge them with a small `CustomJS` callback that recomputes one source from a selection on the other. That is the entire mechanism behind everything below.

**Jupyter integration** is first-class: `output_notebook()` renders inline; `show()` displays a figure or layout; `push_notebook()` even supports live updates from Python. Nothing here requires the separate Bokeh server.

**Why Bokeh over the alternatives**, given the brief excludes Matplotlib, seaborn, and Altair:

- **vs. Plotly** — Plotly renders beautifully and is faster for a first plot, but linking views across *different* data sources typically leans on Dash and a live callback server. Bokeh does cross-source linking in exported static HTML via `CustomJS`. For a self-contained, publishable teaching artifact, that is the deciding factor.
- **vs. Altair** *(excluded anyway)* — Altair's Vega-Lite grammar is elegant but its interactivity ceiling is lower once you need custom cross-source recomputation.
- **vs. Plotly Dash / Streamlit / Panel** — all excellent, all require a running server. That is exactly what we are avoiding.

**The limitations, stated plainly:**

- **`CustomJS` means writing JavaScript.** Non-trivial linking logic lives in JS strings inside your Python — awkward to debug, and no type checking across the boundary.
- **Not built for big data.** Tens of thousands of glyphs in a browser is fine; millions is not, without server-side downsampling (`datashader`) or the Bokeh server.
- **Verbose for simple plots.** The explicitness that pays off for linked dashboards is overhead for a quick histogram.
- **Two moving parts.** Python `bokeh` and the browser `BokehJS` are versioned together; embedding across mismatched versions can bite.

The rule of thumb: reach for Bokeh when interaction is the point. For static figures in a paper, a lighter library wins.

## 5. The dataset: eBird, and cleaning it

[eBird](https://ebird.org), run by the Cornell Lab of Ornithology, is the largest biodiversity dataset in existence — over a billion observations contributed by birders worldwide. It suits this tutorial perfectly because every record carries **three dimensions at once**: *when* (timestamp), *where* (coordinates + named hotspot), and *what* (species). That is exactly the structure that makes linked views sing — each chart can own one dimension while sharing the same rows.

### Getting the data

A free API key comes from https://ebird.org/api/keygen. Two source options, with a trade-off worth knowing:

- **eBird API 2.0** — live, but the recent-observations endpoint reaches back only **30 days** and returns observations without full effort metadata.
- **eBird Basic Dataset (EBD)** — the full monthly research download with effort covariates (duration, distance, protocol, completeness), but released on a lag, so it lacks the most recent weeks.

We pull from the API below. So the notebook runs end-to-end before you have a key, it falls back to a **synthetic sample** with realistic seasonal and habitat structure. Flip `USE_SAMPLE = False` once your key is set.

### Cleaning steps

The raw API rows need five specific fixes, each for a concrete reason:

1. **Parse `obsDt` to datetime.** It arrives as a string; without parsing there is no time axis.
2. **Coerce `howMany` to numeric and flag the gaps.** When an observer logs presence without counting, the count is `"X"` → must become `NaN`, and we keep a `counted` flag rather than silently treating it as zero.
3. **Derive `week`.** Two months is too short for monthly bins and too noisy at daily resolution for trend reading; ISO-week is the right grain.
4. **Keep `subId` (checklist id).** This is the unit of *effort*. We rank and normalize by distinct checklists, not raw rows, so one birder logging a flock of 3,000 geese doesn't dominate.
5. **Drop rows with unparseable dates or missing coordinates** — they can't be placed on the time axis or the map.

> **The caveat that shapes every chart:** eBird is *opportunistic*, not a designed survey. Effort clusters near cities, roads, and weekends. Raw counts confound how many birds there are with how many people went looking. Every panel below either normalizes by checklist volume or is explicitly there to *show* that bias — never to be read as raw abundance.

In [2]:
USE_SAMPLE = True                                   # False once EBIRD_API_KEY is set
import os
EBIRD_API_KEY = os.environ.get("EBIRD_API_KEY", "")

REGION, DAYS_BACK, API_MAX_BACK = "US-MI", 60, 30
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=DAYS_BACK)


def fetch_recent(region=REGION, back=30):
    """eBird recent observations. `back` is capped at 30 days by the API."""
    r = requests.get(
        f"https://api.ebird.org/v2/data/obs/{region}/recent",
        headers={"X-eBirdApiToken": EBIRD_API_KEY},
        params={"back": min(back, API_MAX_BACK)}, timeout=60,
    )
    r.raise_for_status()
    return pd.DataFrame(r.json())

In [3]:
def make_sample(n=9000, seed=42):
    """Synthetic Michigan observations with seasonal + habitat structure.
    For pipeline demonstration only — not real data."""
    rng = np.random.default_rng(seed)
    species = [("American Robin",1.0),("Black-capped Chickadee",.85),("Northern Cardinal",.8),
               ("Blue Jay",.75),("Song Sparrow",.6),("Red-winged Blackbird",.58),
               ("Mourning Dove",.55),("American Goldfinch",.5),("Downy Woodpecker",.42),
               ("White-breasted Nuthatch",.38),("Canada Goose",.36),("Common Grackle",.34),
               ("Tufted Titmouse",.30),("Yellow Warbler",.28),("Baltimore Oriole",.22),
               ("Sandhill Crane",.18),("Great Blue Heron",.16),("Eastern Bluebird",.15),
               ("Indigo Bunting",.12),("Scarlet Tanager",.08)]
    names = [s[0] for s in species]
    w = np.array([s[1] for s in species]); w = w / w.sum()
    spots = [("Nichols Arboretum",42.281,-83.727,"woodland"),("Kensington Metropark",42.518,-83.647,"woodland"),
             ("Erie Marsh Preserve",41.795,-83.452,"wetland"),("Sleeping Bear Dunes",44.882,-86.031,"open"),
             ("Whitefish Point",46.770,-84.958,"open"),("Belle Isle",42.339,-82.983,"woodland"),
             ("Muskegon Wastewater",43.259,-86.026,"wetland"),("Tawas Point",44.254,-83.449,"open"),
             ("Maple River SGA",43.135,-84.700,"wetland"),("Seney NWR",46.243,-85.949,"wetland")]
    wet = {"Red-winged Blackbird","Canada Goose","Great Blue Heron","Sandhill Crane","Song Sparrow"}
    wood = {"Black-capped Chickadee","Downy Woodpecker","Tufted Titmouse",
            "White-breasted Nuthatch","Scarlet Tanager","Blue Jay"}
    days = (END_DATE - START_DATE).days
    offs = rng.integers(0, days + 1, n)
    dates = np.array([START_DATE + timedelta(days=int(o)) for o in offs])
    keep = rng.random(n) < np.where(np.array([d.weekday() for d in dates]) >= 5, .95, .55)
    idx = rng.choice(len(names), n, p=w)
    spot_idx = rng.integers(0, len(spots), n)
    sp = []
    for i, si in zip(idx, spot_idx):
        nm, hab = names[i], spots[si][3]
        if rng.random() < .45:
            pool = wet if hab == "wetland" else wood if hab == "woodland" else set(names)
            nm = rng.choice(sorted(pool))
        sp.append(nm)
    return pd.DataFrame({
        "comName": sp,
        "obsDt": [f"{d} {rng.integers(5,19):02d}:{rng.integers(0,60):02d}" for d in dates],
        "howMany": rng.integers(1, 12, n),
        "locName": [spots[i][0] for i in spot_idx],
        "lat": [spots[i][1] + rng.normal(0,.05) for i in spot_idx],
        "lng": [spots[i][2] + rng.normal(0,.05) for i in spot_idx],
        "subId": [f"S{rng.integers(10**7,10**8)}" for _ in range(n)],
    })[keep].reset_index(drop=True)

In [4]:
# --- Load ---
if USE_SAMPLE or not EBIRD_API_KEY:
    raw, SOURCE = make_sample(), "SYNTHETIC SAMPLE - not real data"
else:
    raw, SOURCE = fetch_recent(back=30), f"eBird API - {REGION} - last 30 days"

# --- Clean (the five steps above) ---
def clean(df):
    d = df.copy()
    d["obsDt"] = pd.to_datetime(d["obsDt"], errors="coerce")     # 1
    d["howMany"] = pd.to_numeric(d.get("howMany"), errors="coerce")  # 2
    d["counted"] = d["howMany"].notna()
    d = d.dropna(subset=["obsDt", "lat", "lng"])                 # 5
    d["week"] = d["obsDt"].dt.to_period("W").dt.start_time        # 3
    d["date"] = d["obsDt"].dt.date
    d["is_weekend"] = d["obsDt"].dt.weekday >= 5
    return d                                                     # 4: subId kept

obs = clean(raw)
print(SOURCE)
print(f"{len(obs):,} rows | {obs['comName'].nunique()} species | "
      f"{obs['subId'].nunique():,} checklists | {obs['locName'].nunique()} locations")
obs.head(3)

SYNTHETIC SAMPLE - not real data
5,934 rows | 20 species | 5,934 checklists | 10 locations


,comName,obsDt,howMany,locName,lat,lng,subId,counted,week,date,is_weekend
0,Tufted Titmouse,2026-05-30 12:19:00,5,Maple River SGA,43.200300,-84.686376,S29310281,True,2026-05-25,2026-05-30,True
1,Blue Jay,2026-07-11 18:26:00,8,Belle Isle,42.333969,-82.918476,S39548804,True,2026-07-06,2026-07-11,True
2,Sandhill Crane,2026-06-20 18:23:00,6,Erie Marsh Preserve,41.770059,-83.400775,S51238116,True,2026-06-15,2026-06-20,True


In [5]:
# Shared aggregates used across the dashboard
TOP_N = 8
species_rank = (obs.groupby("comName")["subId"].nunique()
                .sort_values(ascending=False))
top_species = species_rank.head(TOP_N).index.tolist()

loc_rank = (obs.groupby("locName")
            .agg(lat=("lat","mean"), lng=("lng","mean"),
                 checklists=("subId","nunique"), species=("comName","nunique"))
            .reset_index().sort_values("checklists", ascending=False))

print("Top species:", ", ".join(top_species))

Top species: Black-capped Chickadee, Song Sparrow, Red-winged Blackbird, Blue Jay, Canada Goose, American Robin, Sandhill Crane, Northern Cardinal


## 1. The four visualization types

Each view owns one question. Read the narrative, then the cell that builds it. We construct them individually first, then wire them together in section 2.

### 1a. Temporal — weekly detection rate (multi-line)

A line chart of each top species' **share of that week's checklists**. Lines are the right encoding for time: they make trend and slope pre-attentive, and overlaying species invites comparison of *timing* — who peaks when. We plot a rate, not a count, so a week with more birding doesn't masquerade as more birds.

This view answers **"when?"** and nothing else. It is deliberately blind to location — which is precisely why it needs the map beside it.

In [6]:
from bokeh.models import ColumnDataSource, HoverTool, Legend
from bokeh.palettes import Category10

def weekly_rate_table(frame):
    wk = (frame[frame["comName"].isin(top_species)]
          .groupby(["week","comName"])["subId"].nunique().reset_index(name="det"))
    eff = frame.groupby("week")["subId"].nunique().rename("eff")
    wk = wk.merge(eff, on="week")
    wk["rate"] = wk["det"] / wk["eff"]
    return wk

def make_time_figure(frame):
    wk = weekly_rate_table(frame)
    colors = Category10[max(3, TOP_N)]
    p = figure(height=300, width=560, x_axis_type="datetime",
               tools="pan,wheel_zoom,reset", toolbar_location="above",
               title="When: weekly detection rate by species")
    items = []
    for i, sp in enumerate(top_species):
        g = wk[wk["comName"] == sp].sort_values("week")
        src = ColumnDataSource(dict(week=g["week"], rate=g["rate"],
                                    sp=[sp]*len(g), det=g["det"], eff=g["eff"]))
        r = p.line("week", "rate", source=src, line_width=2.5, color=colors[i])
        p.scatter("week", "rate", source=src, size=6, color=colors[i])
        items.append((sp, [r]))
    p.add_tools(HoverTool(tooltips=[("Species","@sp"),("Week","@week{%b %d}"),
                                    ("Rate","@rate{0.0%}"),("Checklists","@det of @eff")],
                          formatters={"@week":"datetime"}))
    lg = Legend(items=items, label_text_font_size="8pt", location="center")
    p.add_layout(lg, "right")
    p.yaxis.formatter = bokeh.models.NumeralTickFormatter(format="0%")
    p.yaxis.axis_label, p.xaxis.axis_label = "Share of checklists", "Week"
    return p

show(make_time_figure(obs))

### 1b. Spatial — hotspot map (bubble scatter)

Each hotspot is a bubble: **size = checklist volume, color = species richness**, positioned by Web-Mercator coordinates over a tile basemap. A map is non-negotiable for the "where?" question — no bar chart recovers spatial adjacency. The double encoding lets one glance separate a *busy* site from a *diverse* one; they are not the same thing, and the difference is often the story.

This is also the view that makes eBird's central bias **visible**: the bubbles cluster where birders are, not where birds are. Showing that honestly is better than hiding it.

In [7]:
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.palettes import Viridis256

def to_mercator(lat, lon):
    k = 6378137.0
    x = lon * (k * np.pi / 180.0)
    y = np.log(np.tan((90 + lat) * np.pi / 360.0)) * k
    return x, y

def make_map_figure(loc_df):
    d = loc_df.copy()
    d["mx"], d["my"] = to_mercator(d["lat"].values, d["lng"].values)
    d["size"] = 12 + 34 * (d["checklists"] / d["checklists"].max())
    src = ColumnDataSource(d)
    cmap = LinearColorMapper(palette=Viridis256,
                             low=d["species"].min(), high=d["species"].max())
    pad = 120000
    p = figure(height=430, width=560,
               x_range=(d["mx"].min()-pad, d["mx"].max()+pad),
               y_range=(d["my"].min()-pad, d["my"].max()+pad),
               x_axis_type="mercator", y_axis_type="mercator",
               tools="pan,wheel_zoom,box_select,tap,reset", toolbar_location="above",
               title="Where: hotspots (size = checklists, color = richness)")
    p.add_tile("CartoDB Positron")
    p.scatter("mx","my", size="size", source=src,
              fill_color={"field":"species","transform":cmap},
              fill_alpha=0.85, line_color="white", line_width=1.5)
    p.add_tools(HoverTool(tooltips=[("Hotspot","@locName"),("Checklists","@checklists"),
                                    ("Species","@species")]))
    p.add_layout(ColorBar(color_mapper=cmap, title="Richness", width=8), "right")
    return p, src

_mapfig, _ = make_map_figure(loc_rank)
show(_mapfig)

### 1c. Ranking — species frequency (horizontal bars)

A horizontal bar chart ranking species by checklists reporting them. Bars are the most accurate encoding we have for magnitude comparison — position along a common axis beats area, angle, or color every time. Horizontal orientation gives the species labels room to breathe.

On its own this is the least surprising of the four. Its power is entirely **relational**: once linked, it becomes the live answer to "what lives at the places I just selected on the map?" — and *that* is where the habitat structure surfaces.

In [8]:
from bokeh.models import FactorRange

def species_counts(frame, locs=None):
    f = frame if locs is None else frame[frame["locName"].isin(locs)]
    c = (f[f["comName"].isin(top_species)]
         .groupby("comName")["subId"].nunique().reindex(top_species).fillna(0))
    return c

def make_bar_figure(frame):
    c = species_counts(frame)
    order = c.sort_values().index.tolist()          # ascending -> largest on top
    src = ColumnDataSource(dict(sp=order, n=[c[s] for s in order]))
    p = figure(height=300, width=430, y_range=FactorRange(*order),
               tools="", toolbar_location=None,
               title="What: species by checklists (linked to map)")
    p.hbar(y="sp", right="n", height=0.78, source=src,
           fill_color="#2a9d8f", line_color=None)
    p.add_tools(HoverTool(tooltips=[("Species","@sp"),("Checklists","@n{0,0}")]))
    p.xaxis.axis_label = "Checklists"
    return p, src

_barfig, _ = make_bar_figure(obs)
show(_barfig)

### 1d. Effort context — daily volume (bars)

Daily checklist counts, weekends highlighted. This is the **denominator made visible**. Before trusting any rise in the line chart, you check here whether that week was thinly or heavily sampled. It is the humblest panel and the one that keeps the other three honest — a spike in detections during a week of low effort is noise, and only this view tells you so.

In [9]:
def make_effort_figure(frame):
    d = (frame.groupby("date")
         .agg(checklists=("subId","nunique")).reset_index())
    d["date"] = pd.to_datetime(d["date"])
    d["wknd"] = d["date"].dt.weekday >= 5
    d["color"] = np.where(d["wknd"], "#e07a5f", "#a8b8c8")
    src = ColumnDataSource(d)
    p = figure(height=200, width=1000, x_axis_type="datetime",
               tools="pan,wheel_zoom,reset", toolbar_location="above",
               title="Effort context: checklists per day (orange = weekend)")
    p.vbar(x="date", top="checklists", width=7e7, source=src, color="color")
    p.add_tools(HoverTool(tooltips=[("Date","@date{%a %b %d}"),("Checklists","@checklists")],
                          formatters={"@date":"datetime"}))
    p.yaxis.axis_label = "Checklists"
    return p

show(make_effort_figure(obs))

## 2. How they complement each other — and the interactivity that binds them

The four views were chosen to span eBird's three axes with no redundancy, plus a denominator:

| View | Encoding | Axis | Answers | Blind to |
|------|----------|------|---------|----------|
| Time lines | line | time | *when* species appear | where |
| Hotspot map | bubble | space | *where* effort concentrates | when, which species |
| Species bars | bar | species | *what* is most reported | when, where |
| Effort bars | bar | time | *how much* sampling | species, space |

Each is weakest exactly where the next is strongest. But the real payoff is **coordination**, and it comes in two flavors Bokeh handles differently:

**Shared-source linking (free).** Glyphs backed by one `ColumnDataSource` share a selection automatically. Brush a set of points and they highlight everywhere that source is drawn — no callback required.

**Cross-source linking (a small `CustomJS` bridge).** Our map and bar chart have *different* sources — one row per hotspot vs. one row per species. So we attach a callback: when the map's selection changes, recompute the species tallies from just the selected hotspots and push them into the bar source. **Select a cluster of wetland sites and the bars re-sort to Red-winged Blackbird and waterfowl; select woodland sites and chickadees and nuthatches rise.** The bar chart stops being a static ranking and becomes a live readout of *"what lives where I'm looking."*

That single interaction is the thesis of the whole tutorial: the insight lives in the *link*, not in any one chart.

### Dashboard-specific considerations

- **One shared selection color** and consistent species colors across panels, so the eye tracks entities between views.
- **A default (nothing-selected) state that shows the global totals** — an empty selection means "all," never a blank chart.
- **Layout encodes reading order**: effort spanning the bottom as a foundation, the "where→what" pair (map + bars) side by side because that is the interaction the user drives, time on top for context.
- **Tool choice is deliberate**: box- and tap-select on the map (the driver); pan/zoom only on the time and effort views (context, not drivers). Giving every chart every tool is a common way to make a dashboard feel noisy.

Now the bridge:

In [10]:
from bokeh.models import CustomJS

# One obs-per-(species,location) matrix, shipped to the browser so JS can
# re-tally species for any subset of selected hotspots.
matrix = (obs[obs["comName"].isin(top_species)]
          .groupby(["locName","comName"])["subId"].nunique()
          .reset_index(name="n"))
matrix_src = ColumnDataSource(matrix)

def link_map_to_bars(map_src, bar_src, matrix_src, species_order):
    cb = CustomJS(
        args=dict(map_src=map_src, bar_src=bar_src, M=matrix_src, order=species_order),
        code="""
        const sel = map_src.selected.indices;
        const locs = map_src.data['locName'];
        const chosen = sel.length ? sel.map(i => locs[i]) : locs;   // empty = all
        const md = M.data;
        const tot = {};
        for (const s of order) tot[s] = 0;
        for (let i = 0; i < md['locName'].length; i++) {
            if (chosen.includes(md['locName'][i])) tot[md['comName'][i]] += md['n'][i];
        }
        // re-sort ascending so the largest bar sits on top, matching the static build
        const sorted = [...order].sort((a,b) => tot[a] - tot[b]);
        bar_src.data['sp'] = sorted;
        bar_src.data['n']  = sorted.map(s => tot[s]);
        bar_src.change.emit();
    """)
    map_src.selected.js_on_change("indices", cb)

print("Bridge defined. Matrix rows:", len(matrix))

Bridge defined. Matrix rows: 80


### Assembling the linked dashboard

We rebuild the map and bars so they share the live sources the bridge updates, wire them, and lay all four out in a grid. The result below is fully interactive: **drag a box across hotspots on the map and watch the species bars recompute.**

In [11]:
from bokeh.layouts import gridplot, column as bk_column
from bokeh.models import Div

def build_dashboard(frame, loc_df):
    time_fig = make_time_figure(frame)
    map_fig, map_src = make_map_figure(loc_df)
    bar_fig, bar_src = make_bar_figure(frame)
    eff_fig = make_effort_figure(frame)

    # order must match make_bar_figure's ascending sort for the initial state
    init = species_counts(frame).sort_values().index.tolist()
    link_map_to_bars(map_src, bar_src, matrix_src, init)

    header = Div(text=f"""
        <div style='font-family:sans-serif'>
        <h2 style='margin:0 0 4px'>eBird Michigan - linked exploration</h2>
        <p style='margin:0;color:#666;font-size:13px'>{SOURCE} &nbsp;|&nbsp;
        {frame['subId'].nunique():,} checklists &nbsp;|&nbsp;
        <b>Box-select hotspots on the map -&gt; species bars update.</b></p></div>""",
        width=1000)

    grid = gridplot([[time_fig, None],[map_fig, bar_fig],[eff_fig, None]],
                    toolbar_location="right", merge_tools=True)
    return bk_column(header, grid)

dashboard = build_dashboard(obs, loc_rank)
show(dashboard)

## Export: one self-contained, interactive file

`file_html` inlines BokehJS and the scene graph into a single HTML document. No server, no build step — the linked selection keeps working for anyone who opens it. This is the property that made Bokeh the right pick for a *shareable* teaching artifact.

In [12]:
from bokeh.embed import file_html
from bokeh.resources import INLINE

html = file_html(dashboard, INLINE, "eBird Michigan - Linked Dashboard")
with open("ebird_dashboard.html", "w", encoding="utf-8") as f:
    f.write(html)
print(f"Wrote ebird_dashboard.html ({len(html)/1e6:.1f} MB, self-contained)")

Wrote ebird_dashboard.html (1.7 MB, self-contained)


## Takeaways

**The design lesson.** Coordinated views change how you build each chart. A bar chart that would be dull in isolation becomes the payoff of the dashboard once it responds to a spatial selection. Design the *linkage* first, then each view to serve it.

**When to reach for this.** Coordinated multiple views earn their complexity for *exploratory* analysis — when the questions are open-ended and you want to slice by one dimension and watch the others respond. For a fixed narrative with one message, a single well-made chart is better; don't add interactivity as decoration.

**Why Bokeh here specifically.** Cross-source linking that survives export to static HTML. Plotly's first plot is prettier, and for server-backed apps Dash or Streamlit or Panel may fit better — but for a self-contained, interactive artifact others can open and learn from, Bokeh's client-side model is hard to beat.

**Honest limits, carried throughout.** eBird is opportunistic; the effort panel exists to keep that visible. For real analysis, move to the EBD for effort covariates, add year-over-year comparison so two months aren't read in isolation, and spatially subsample to blunt the clustering the map makes so plain. And check eBird's [terms of use](https://www.birds.cornell.edu/home/ebird-api-terms-of-use/) before republishing records — some sensitive species have obscured locations to protect nests.

*Data: eBird, Cornell Lab of Ornithology. Basemap (c) CartoDB, OpenStreetMap contributors. Built with Bokeh (BSD-3).*